# Module 11 Lab: Real-Time Object Tracking and Zone-Based Analytics

Welcome to an advanced topic in computer vision! In this lab, you will learn how to not just detect objects, but **track** them across video frames. You will also learn how to perform **zone-based analytics**, which involves analyzing object behavior within a specific region of the video.

**What you'll learn:**
- The difference between object detection and object tracking.
- How to use a pre-trained model to track objects in a video.
- How to define a 'zone' and count objects that enter it.

## Learning Objectives

By the end of this lab, you will be able to:

- **Differentiate** between object detection and object tracking.
- **Implement** an object tracking pipeline using the YOLO model.
- **Design** and implement a zone for analytics.
- **Analyze** the behavior of objects within a defined zone.

## 1. Setup and Installation

We will continue to use the `ultralytics` and `opencv-python` libraries.


In [1]:
%pip install ultralytics opencv-python requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.6 MB/s eta 0:00:00


Now, let's import the necessary components.

In [2]:
from ultralytics import YOLO
import cv2
import requests
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## 2. Understanding Object Tracking

### Object Detection vs. Object Tracking

- **Object Detection** is the process of identifying and locating objects in a single image or frame. It answers the question, 'What objects are in this frame and where are they?'
- **Object Tracking** is the process of identifying an object and following it across multiple frames. It answers the question, 'Where is this specific object going?' Tracking assigns a unique ID to each detected object, allowing it to be followed over time.

### Why is Tracking Useful?

Tracking is essential for many real-world applications, such as:
- **Retail Analytics:** Counting how many people enter a store and where they go.
- **Traffic Management:** Monitoring the flow of cars on a highway.
- **Security:** Following a person of interest through a building.

## Part 1: Coded Demonstration

In this part, we will track cars in a video and count how many of them enter a specific zone.

In [3]:
# 1. Load the YOLO model
model = YOLO('yolov8n.pt')

In [4]:
# 2. Download a video to analyze
video_url = 'https://archive.org/download/highway_construction_2/highway_construction_2_512kb.mp4'
video_path = 'highway.mp4'
r = requests.get(video_url, stream=True)
with open(video_path, 'wb') as f:
    for chunk in r.iter_content(chunk_size=1024*1024):
        if chunk:
            f.write(chunk)

In [5]:
# 3. Create the Tracking and Zone Analytics Agent

def tracking_agent(video_path):
    cap = cv2.VideoCapture(video_path)

    # Define the zone (a horizontal line in this case)
    zone_y = 250

    # Store the IDs of objects that have crossed the zone
    crossed_objects = set()

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Track objects in the frame
        results = model.track(frame, persist=True)

        # Get the annotated frame with bounding boxes and track IDs
        annotated_frame = results[0].plot()

        # Draw the zone line
        cv2.line(annotated_frame, (0, zone_y), (frame.shape[1], zone_y), (0, 255, 255), 2)

        # Check if any objects have crossed the zone
        if results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu()
            track_ids = results[0].boxes.id.int().cpu().tolist()

            for box, track_id in zip(boxes, track_ids):
                # Check if the bottom of the box has crossed the line
                if box[3] > zone_y and track_id not in crossed_objects:
                    crossed_objects.add(track_id)
                    print(f'Object with ID {track_id} crossed the zone!')

        # Display the total count of crossed objects
        cv2.putText(annotated_frame, f'Crossed: {len(crossed_objects)}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)

        # Display the frame
        img = Image.fromarray(cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB))
        # To display in a notebook, we can show a few frames
        if cap.get(cv2.CAP_PROP_POS_FRAMES) % 50 == 0:
            plt.imshow(img)
            plt.show()

    cap.release()

# Run the agent
tracking_agent(video_path)

## Part 2: Student Challenge

Your challenge is to modify the agent to count objects in a different zone. Instead of a horizontal line, you will define a **rectangular zone** and count how many objects are **inside** it in each frame.

**Your Task:**
1.  Define a rectangular zone (e.g., `zone = [x1, y1, x2, y2]`).
2.  Modify the agent to count how many objects are inside the zone in each frame.
3.  Display the count on the frame.

In [8]:
# 3. Define rectangular zone
zone = [200, 150, 500, 350]  # x1, y1, x2, y2

# 4. Agent function
def zone_counting_agent(video_path):
    cap = cv2.VideoCapture(video_path)

    zx1, zy1, zx2, zy2 = zone

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        results = model.track(frame, persist=True)
        annotated_frame = results[0].plot()

        # Draw zone
        cv2.rectangle(annotated_frame, (zx1, zy1), (zx2, zy2), (0, 255, 255), 2)

        count_in_zone = 0

        if results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu()

            for box in boxes:
                x1, y1, x2, y2 = map(int, box)

                # center point of object
                cx = (x1 + x2) // 2
                cy = (y1 + y2) // 2

                if zx1 <= cx <= zx2 and zy1 <= cy <= zy2:
                    count_in_zone += 1
                    color = (0, 255, 0)
                else:
                    color = (0, 0, 255)

                cv2.circle(annotated_frame, (cx, cy), 5, color, -1)

        # Make the count very visible
        cv2.rectangle(annotated_frame, (10, 10), (320, 70), (0, 0, 0), -1)
        cv2.putText(
            annotated_frame,
            f'Objects in Zone: {count_in_zone}',
            (20, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 255),
            3
        )

        # Show every 50th frame
        if int(cap.get(cv2.CAP_PROP_POS_FRAMES)) % 50 == 0:
            img = Image.fromarray(cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB))
            plt.figure(figsize=(12, 7))
            plt.imshow(img)
            plt.axis('off')
            plt.show()

    cap.release()

# 5. Run agent
zone_counting_agent(video_path)

## Reflective Questions

Please answer the following questions in a new Markdown cell below.

1.  What is the key difference between `model.predict()` and `model.track()` in the YOLO library?
2.  In our coded demonstration, we only counted an object once when it crossed the line. How did we achieve this? Why is it important?
3.  Describe a real-world application for zone-based analytics. What kind of zone would you define, and what would you measure?
4.  What are some of the challenges in object tracking? (e.g., what happens if an object is hidden for a few frames?)

## Reflection Questions

### 1. Difference between `model.predict()` and `model.track()`
The key difference between `model.predict()` and `model.track()` in YOLO is that `predict()` only detects objects in a single frame, while `track()` both detects and assigns a unique ID to each object across multiple frames. This means `predict()` treats every frame independently, but `track()` follows the same object over time. Tracking is important when you want to analyze movement, count objects, or understand behavior instead of just identifying what is present in one image.

---

### 2. Counting Objects Only Once
In the demonstration, we counted an object only once by storing its **track ID** in a set (e.g., `crossed_objects`). When an object crossed the line, we checked if its ID was already in the set. If it was not, we added it and counted it.  

This is important because without tracking IDs, the same object could be counted multiple times across different frames, leading to inaccurate results. Using unique IDs ensures each object is only counted once, making the counting system reliable.

---

### 3. Real-World Application of Zone-Based Analytics
A real-world example of zone-based analytics is in **retail stores**. A store could define a rectangular zone around a display area and measure how many customers enter that space. This could help analyze customer interest in certain products.  

Another example is **traffic monitoring**, where a zone could be placed at an intersection to count how many vehicles pass through a specific area. This data could be used to improve traffic flow or adjust signal timing.

---

### 4. Challenges in Object Tracking
Object tracking comes with several challenges. One major issue is **occlusion**, where an object is partially or fully hidden behind another object for a few frames. When this happens, the tracking system might lose the object and assign it a new ID when it reappears.  

Other challenges include fast movement, poor lighting, and overlapping objects. These situations can cause incorrect tracking or missed detections. Maintaining consistent tracking across frames is difficult, especially in crowded or complex environments.

## Submission Instructions

1.  Complete the **Student Challenge** section with your modified agent.
2.  Answer the **Reflective Questions** in a new Markdown cell.
3.  Save your completed notebook (`.ipynb` file) and submit it.